# Statistical Arbitrage - Intraday Analysis (Fixed)

This notebook tests the stat arb system with:
1. **Fixed transaction cost calculation** - based on actual notional, not average price
2. **Matched timeframes** - hedge ratios calculated on same timeframe as trading
3. **15min and 1hour data** - better for mean-reversion than 1min

## Key Changes from Original:
- Transaction costs now correctly calculated as: `(entry_notional + exit_notional) * bps / 10000`
- Cointegration test runs on 15min/1hour data, not daily
- Z-score window adjusted for timeframe (60 bars = 15 hours for 15min data)

In [1]:
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import polars as pl
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import numpy as np

from utils.config import CONFIG
from analysis.preprocessing import load_processed
from analysis.cointegration import find_cointegrated_pairs, save_cointegration_results
from analysis.spread import build_spread_frame
from strategy.backtest import backtest_all_pairs, save_trades
from strategy.portfolio import (
    build_equity_curve,
    pair_performance_summary,
)
from utils.metrics import calculate_all_metrics

print(f"Project root: {project_root}")

Project root: /Users/erevtsov/dev/stat-arb


## Test 1: 15-Minute Timeframe

In [2]:
# Run cointegration on 15min data
# Adjust half-life thresholds for 15min bars:
# - 1 day = 26 bars (6.5 trading hours * 4 bars/hour)
# - 5 days = 130 bars
# - 30 days = ~780 bars

print("Finding cointegrated pairs on 15min data...")
coint_15min = find_cointegrated_pairs(
    timeframe="15min",
    min_half_life=50,     # ~2 days
    max_half_life=1000,   # ~38 days
    p_value_threshold=0.05
)

print(f"Found {len(coint_15min)} cointegrated pairs")
print("\nTop 10 pairs:")
print(coint_15min.head(10))

Finding cointegrated pairs on 15min data...


Testing cointegration: 100%|██████████| 411/411 [00:01<00:00, 334.96it/s]

Found 129 cointegrated pairs

Top 10 pairs:
shape: (10, 8)
┌──────────┬──────────┬─────────────┬───────────┬──────────┬────────────┬────────────┬───────────┐
│ ticker_a ┆ ticker_b ┆ hedge_ratio ┆ adf_stat  ┆ p_value  ┆ half_life  ┆ sector     ┆ timeframe │
│ ---      ┆ ---      ┆ ---         ┆ ---       ┆ ---      ┆ ---        ┆ ---        ┆ ---       │
│ str      ┆ str      ┆ f64         ┆ f64       ┆ f64      ┆ f64        ┆ str        ┆ str       │
╞══════════╪══════════╪═════════════╪═══════════╪══════════╪════════════╪════════════╪═══════════╡
│ PFE      ┆ ABBV     ┆ -0.153242   ┆ -4.527319 ┆ 0.000175 ┆ 440.97851  ┆ Healthcare ┆ 15min     │
│ PFE      ┆ LLY      ┆ -0.022136   ┆ -4.453451 ┆ 0.000238 ┆ 377.50289  ┆ Healthcare ┆ 15min     │
│ BLK      ┆ PNC      ┆ 4.258573    ┆ -4.369602 ┆ 0.000336 ┆ 73.18039   ┆ Financials ┆ 15min     │
│ PFE      ┆ UNH      ┆ -0.048591   ┆ -4.332569 ┆ 0.00039  ┆ 501.632754 ┆ Healthcare ┆ 15min     │
│ DUK      ┆ SO       ┆ 1.059449    ┆ -4.27956  ┆ 

In [3]:
# Backtest on 15min data with adjusted parameters
# Z-score window: 60 bars = 15 hours of trading
# Transaction costs: 5 bps per leg (more realistic for large-cap stocks)

if len(coint_15min) > 0:
    print("\nBacktesting top 10 pairs on 15min data...")
    trades_15min = backtest_all_pairs(
        coint_15min,
        timeframe="15min",
        max_pairs=10,
        z_entry=2.5,
        z_stop=4.0,
        max_holding_minutes=6.5*60,  # 1 trading day
        zscore_window=60,             # 15 hours
        transaction_cost_bps=5,       # 5 bps per leg (20 bps round-trip total)
    )
    
    print(f"\nTotal trades: {len(trades_15min)}")
    
    if len(trades_15min) > 0:
        # Calculate metrics
        pair_perf = pair_performance_summary(trades_15min)
        print("\nPer-pair performance:")
        print(pair_perf)
        
        equity_15min = build_equity_curve(trades_15min)
        metrics_15min = calculate_all_metrics(trades_15min, equity_15min["equity"])
        
        print("\n=== 15min Performance Metrics ===")
        for k, v in metrics_15min.items():
            if k != "exit_reason_breakdown":
                print(f"  {k:25s}: {v}")
else:
    print("No cointegrated pairs found on 15min timeframe")


Backtesting top 10 pairs on 15min data...


Backtesting pairs: 100%|██████████| 10/10 [00:00<00:00, 85.87it/s]



Total trades: 1251

Per-pair performance:
shape: (10, 6)
┌──────────┬────────────┬──────────┬──────────┬────────────┬─────────────────┐
│ pair     ┆ num_trades ┆ win_rate ┆ avg_pnl  ┆ total_pnl  ┆ avg_holding_min │
│ ---      ┆ ---        ┆ ---      ┆ ---      ┆ ---        ┆ ---             │
│ str      ┆ u32        ┆ f64      ┆ f64      ┆ f64        ┆ f64             │
╞══════════╪════════════╪══════════╪══════════╪════════════╪═════════════════╡
│ BLK/PNC  ┆ 110        ┆ 0.663636 ┆ 6.89275  ┆ 758.202529 ┆ 1575.409091     │
│ PSX/VLO  ┆ 109        ┆ 0.59633  ┆ 2.461323 ┆ 268.284235 ┆ 1470.550459     │
│ MPC/VLO  ┆ 112        ┆ 0.589286 ┆ 1.903301 ┆ 213.16973  ┆ 1497.1875       │
│ DUK/SO   ┆ 118        ┆ 0.652542 ┆ 0.708113 ┆ 83.55734   ┆ 1872.076271     │
│ PFE/AMGN ┆ 132        ┆ 0.484848 ┆ 0.122884 ┆ 16.220654  ┆ 1177.272727     │
│ PFE/ABBV ┆ 144        ┆ 0.423611 ┆ 0.095102 ┆ 13.694755  ┆ 1023.020833     │
│ PFE/LLY  ┆ 148        ┆ 0.425676 ┆ 0.08799  ┆ 13.022501  ┆ 890.878378  

In [4]:
# Plot equity curve for 15min
if len(coint_15min) > 0 and len(trades_15min) > 0:
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=equity_15min["timestamp"].to_list(),
        y=equity_15min["equity"].to_list(),
        name="Equity",
        line=dict(width=2),
    ))
    fig.add_hline(y=CONFIG["capital"], line_dash="dot", line_color="gray",
                  annotation_text="Starting Capital")
    fig.update_layout(
        title="15min Timeframe - Equity Curve",
        xaxis_title="Time",
        yaxis_title="Equity ($)",
        height=500,
    )
    fig.show()

## Test 2: 1-Hour Timeframe

In [5]:
# Run cointegration on 1hour data
# Adjust half-life thresholds for 1hour bars:
# - 1 day = 6.5 bars
# - 5 days = ~32 bars
# - 30 days = ~195 bars

print("Finding cointegrated pairs on 1hour data...")
coint_1hour = find_cointegrated_pairs(
    timeframe="1hour",
    min_half_life=10,     # ~1.5 days
    max_half_life=250,    # ~38 days
    p_value_threshold=0.05
)

print(f"Found {len(coint_1hour)} cointegrated pairs")
print("\nTop 10 pairs:")
print(coint_1hour.head(10))

Finding cointegrated pairs on 1hour data...


Testing cointegration: 100%|██████████| 411/411 [00:00<00:00, 592.80it/s]

Found 94 cointegrated pairs

Top 10 pairs:
shape: (10, 8)
┌──────────┬──────────┬─────────────┬───────────┬──────────┬────────────┬──────────────┬───────────┐
│ ticker_a ┆ ticker_b ┆ hedge_ratio ┆ adf_stat  ┆ p_value  ┆ half_life  ┆ sector       ┆ timeframe │
│ ---      ┆ ---      ┆ ---         ┆ ---       ┆ ---      ┆ ---        ┆ ---          ┆ ---       │
│ str      ┆ str      ┆ f64         ┆ f64       ┆ f64      ┆ f64        ┆ str          ┆ str       │
╞══════════╪══════════╪═════════════╪═══════════╪══════════╪════════════╪══════════════╪═══════════╡
│ MPC      ┆ PSX      ┆ 1.238002    ┆ -4.768305 ┆ 0.000063 ┆ 18.865904  ┆ Energy       ┆ 1hour     │
│ PFE      ┆ ABBV     ┆ -0.160052   ┆ -4.321213 ┆ 0.000408 ┆ 162.746991 ┆ Healthcare   ┆ 1hour     │
│ PFE      ┆ TMO      ┆ -0.009797   ┆ -4.282768 ┆ 0.000476 ┆ 220.741461 ┆ Healthcare   ┆ 1hour     │
│ PFE      ┆ LLY      ┆ -0.02341    ┆ -4.155357 ┆ 0.000783 ┆ 126.593307 ┆ Healthcare   ┆ 1hour     │
│ PFE      ┆ ABT      ┆ -0.246524

In [6]:
# Backtest on 1hour data
# Z-score window: 60 bars = ~9 trading days

if len(coint_1hour) > 0:
    print("\nBacktesting top 10 pairs on 1hour data...")
    trades_1hour = backtest_all_pairs(
        coint_1hour,
        timeframe="1hour",
        max_pairs=10,
        z_entry=2.5,
        z_stop=4.0,
        max_holding_minutes=3*6.5*60,  # 3 trading days
        zscore_window=60,               # ~9 trading days
        transaction_cost_bps=5,         # 5 bps per leg
    )
    
    print(f"\nTotal trades: {len(trades_1hour)}")
    
    if len(trades_1hour) > 0:
        # Calculate metrics
        pair_perf = pair_performance_summary(trades_1hour)
        print("\nPer-pair performance:")
        print(pair_perf)
        
        equity_1hour = build_equity_curve(trades_1hour)
        metrics_1hour = calculate_all_metrics(trades_1hour, equity_1hour["equity"])
        
        print("\n=== 1hour Performance Metrics ===")
        for k, v in metrics_1hour.items():
            if k != "exit_reason_breakdown":
                print(f"  {k:25s}: {v}")
else:
    print("No cointegrated pairs found on 1hour timeframe")


Backtesting top 10 pairs on 1hour data...


Backtesting pairs: 100%|██████████| 10/10 [00:00<00:00, 161.11it/s]


Total trades: 460

Per-pair performance:
shape: (10, 6)
┌──────────┬────────────┬──────────┬───────────┬────────────┬─────────────────┐
│ pair     ┆ num_trades ┆ win_rate ┆ avg_pnl   ┆ total_pnl  ┆ avg_holding_min │
│ ---      ┆ ---        ┆ ---      ┆ ---       ┆ ---        ┆ ---             │
│ str      ┆ u32        ┆ f64      ┆ f64       ┆ f64        ┆ f64             │
╞══════════╪════════════╪══════════╪═══════════╪════════════╪═════════════════╡
│ BLK/PNC  ┆ 40         ┆ 0.65     ┆ 7.039409  ┆ 281.57636  ┆ 1381.5          │
│ GS/PNC   ┆ 38         ┆ 0.552632 ┆ 3.555153  ┆ 135.095821 ┆ 1460.526316     │
│ MPC/PSX  ┆ 46         ┆ 0.413043 ┆ 2.108876  ┆ 97.008294  ┆ 1022.608696     │
│ PFE/AMGN ┆ 50         ┆ 0.46     ┆ 0.158537  ┆ 7.926835   ┆ 940.8           │
│ PFE/LLY  ┆ 56         ┆ 0.482143 ┆ 0.099503  ┆ 5.572167   ┆ 1264.285714     │
│ PFE/ABT  ┆ 41         ┆ 0.390244 ┆ 0.115502  ┆ 4.735565   ┆ 1202.926829     │
│ PFE/ABBV ┆ 50         ┆ 0.38     ┆ 0.075733  ┆ 3.786673   ┆ 1

In [7]:
# Plot equity curve for 1hour
if len(coint_1hour) > 0 and len(trades_1hour) > 0:
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=equity_1hour["timestamp"].to_list(),
        y=equity_1hour["equity"].to_list(),
        name="Equity",
        line=dict(width=2),
    ))
    fig.add_hline(y=CONFIG["capital"], line_dash="dot", line_color="gray",
                  annotation_text="Starting Capital")
    fig.update_layout(
        title="1hour Timeframe - Equity Curve",
        xaxis_title="Time",
        yaxis_title="Equity ($)",
        height=500,
    )
    fig.show()

## Comparison: Original (1min) vs Fixed (15min/1hour)

In [8]:
# Load original results for comparison
results_dir = Path(CONFIG["results_dir"])
original_trades_file = results_dir / "trades.parquet"

comparison_data = []

# Original (1min with daily hedge ratios)
if original_trades_file.exists():
    trades_orig = pl.read_parquet(original_trades_file)
    equity_orig = build_equity_curve(trades_orig)
    metrics_orig = calculate_all_metrics(trades_orig, equity_orig["equity"])
    comparison_data.append({
        "timeframe": "1min (original)",
        "total_trades": metrics_orig["total_trades"],
        "win_rate": f"{metrics_orig['win_rate']*100:.2f}%",
        "total_pnl": f"${metrics_orig['total_pnl']:.2f}",
        "sharpe_ratio": f"{metrics_orig['sharpe_ratio']:.2f}",
        "max_drawdown": f"{metrics_orig['max_drawdown']*100:.2f}%",
    })

# 15min
if len(coint_15min) > 0 and len(trades_15min) > 0:
    comparison_data.append({
        "timeframe": "15min (fixed)",
        "total_trades": metrics_15min["total_trades"],
        "win_rate": f"{metrics_15min['win_rate']*100:.2f}%",
        "total_pnl": f"${metrics_15min['total_pnl']:.2f}",
        "sharpe_ratio": f"{metrics_15min['sharpe_ratio']:.2f}",
        "max_drawdown": f"{metrics_15min['max_drawdown']*100:.2f}%",
    })

# 1hour
if len(coint_1hour) > 0 and len(trades_1hour) > 0:
    comparison_data.append({
        "timeframe": "1hour (fixed)",
        "total_trades": metrics_1hour["total_trades"],
        "win_rate": f"{metrics_1hour['win_rate']*100:.2f}%",
        "total_pnl": f"${metrics_1hour['total_pnl']:.2f}",
        "sharpe_ratio": f"{metrics_1hour['sharpe_ratio']:.2f}",
        "max_drawdown": f"{metrics_1hour['max_drawdown']*100:.2f}%",
    })

if comparison_data:
    comparison_df = pl.DataFrame(comparison_data)
    print("\n" + "="*80)
    print("PERFORMANCE COMPARISON")
    print("="*80)
    print(comparison_df)
    print("\nKey Improvements:")
    print("1. Fixed transaction cost calculation (now based on actual notional)")
    print("2. Hedge ratios calculated on same timeframe as trading")
    print("3. Using longer timeframes (15min/1hour) for better mean-reversion")
    print("4. Reduced transaction costs to 5bps per leg (vs 20bps original)")


PERFORMANCE COMPARISON
shape: (3, 6)
┌─────────────────┬──────────────┬──────────┬────────────┬──────────────┬──────────────┐
│ timeframe       ┆ total_trades ┆ win_rate ┆ total_pnl  ┆ sharpe_ratio ┆ max_drawdown │
│ ---             ┆ ---          ┆ ---      ┆ ---        ┆ ---          ┆ ---          │
│ str             ┆ i64          ┆ str      ┆ str        ┆ str          ┆ str          │
╞═════════════════╪══════════════╪══════════╪════════════╪══════════════╪══════════════╡
│ 1min (original) ┆ 9349         ┆ 2.45%    ┆ $-15719.52 ┆ -21.80       ┆ -15.72%      │
│ 15min (fixed)   ┆ 1251         ┆ 49.00%   ┆ $1389.96   ┆ 3.71         ┆ -0.02%       │
│ 1hour (fixed)   ┆ 460          ┆ 44.35%   ┆ $538.41    ┆ 3.97         ┆ -0.01%       │
└─────────────────┴──────────────┴──────────┴────────────┴──────────────┴──────────────┘

Key Improvements:
1. Fixed transaction cost calculation (now based on actual notional)
2. Hedge ratios calculated on same timeframe as trading
3. Using longer t

## Next Steps

If results are still poor:
1. **Increase z-score entry threshold** (try 3.0 or 3.5) - be more selective
2. **Add volume filters** - only trade during liquid hours (9:45-15:30)
3. **Walk-forward validation** - calculate hedge ratios on rolling window
4. **Dynamic hedge ratio** - use Kalman filter for time-varying hedge ratio
5. **Add volatility regime filter** - skip trading during high-vol periods
6. **Try daily timeframe** - pairs trading often works better at daily frequency
7. **Check regime** - 2023-2024 might not be a good period for this strategy